# Evaluating and Optimizing Prompts with DSPy

## Welcome!

Imagine running an LLM-based system that summarizes thousands of meeting transcripts every day. Hand-writing prompts is fragile, since every adjustment to handle one edge case risks breaking others. That's why it's crucial to evaluate and improve prompts programmatically.

In this module, you'll use **DSPy** to take a different approach: instead of writing prompts, you'll *compile* them. You declare what you want, define a metric that scores quality, and let an optimizer find the best prompts automatically.

### What You'll Build

A meeting transcript summarizer that extracts:
1. **Key decisions** made during the meeting
2. **Action items** with owners
3. **A concise summary** of the discussion

### What You'll Learn

- How DSPy signatures replace hand-written prompts
- How to evaluate LLM outputs systematically with metrics
- How `BootstrapFewShot` optimization finds better prompts than you can write by hand
- The "compile, don't write" philosophy for production GenAI

## 1. Install Dependencies

DSPy handles the LLM orchestration and optimization. We use `boto3` for Bedrock authentication.

In [ ]:
!pip install dspy boto3 --quiet

## 2. Configure DSPy with Amazon Bedrock

DSPy supports Bedrock natively via LiteLLM. We configure it with a single line — no API keys needed if your AWS credentials are set up.

Let's verify the connection works with a quick test call.

In [ ]:
import dspy

lm = dspy.LM("bedrock/global.anthropic.claude-haiku-4-5-20251001-v1:0")
dspy.configure(lm=lm)

# Quick sanity check
response = lm("Say 'hello' and nothing else.")
print(f"Connection OK: {response}")

## 3. Load the Meeting Data

We have 15 synthetic meeting transcripts covering different meeting types (standups, planning, retros, etc.). Each example includes the transcript and expected outputs: decisions, action items, and a summary.

Let's load them and take a look at one.

In [ ]:
import json

with open("data/meetings.json") as f:
    meetings = json.load(f)

print(f"Loaded {len(meetings)} meeting examples")
print(f"Meeting types: {[m['meeting_type'] for m in meetings]}")

# Show one example
example = meetings[0]
print(f"\n--- Example: {example['meeting_type']} ---")
print(f"\nTranscript (first 300 chars):\n{example['transcript'][:300]}...")
print(f"\nExpected decisions: {example['expected_decisions']}")
print(f"Expected action items: {example['expected_action_items']}")
print(f"Expected summary: {example['expected_summary']}")

## 4. Prepare DSPy Examples

DSPy uses `Example` objects to represent training and test data. We convert our meeting data into this format and split into **10 training** / **5 test** examples.

The `with_inputs()` call tells DSPy which fields are inputs (the transcript) vs expected outputs (everything else).

In [ ]:
dataset = [
    dspy.Example(
        transcript=m["transcript"],
        decisions=m["expected_decisions"],
        action_items=m["expected_action_items"],
        summary=m["expected_summary"],
    ).with_inputs("transcript")
    for m in meetings
]

trainset = dataset[:10]
testset = dataset[10:]

print(f"Training examples: {len(trainset)}")
print(f"Test examples: {len(testset)}")
print(f"\nInput fields: {trainset[0].inputs().keys()}")
print(f"Output fields: {trainset[0].labels().keys()}")

## 5. Define a DSPy Signature

A **Signature** is DSPy's core abstraction — it defines the input/output contract for an LLM call. Think of it like a function signature, but for prompts.

Instead of writing a prompt like *"You are a meeting assistant that..."*, you declare **what** you want and let DSPy figure out **how** to ask for it.

The docstring becomes the instruction, and the field descriptions guide the model's output format.

In [ ]:
class MeetingSummary(dspy.Signature):
    """Summarize a meeting transcript by extracting key decisions, action items, and a concise summary."""

    transcript: str = dspy.InputField(desc="Full meeting transcript with speaker names")
    decisions: list[str] = dspy.OutputField(desc="Key decisions made during the meeting")
    action_items: list[str] = dspy.OutputField(desc="Action items with owners, e.g. 'Alice: Update the docs'")
    summary: str = dspy.OutputField(desc="2-3 sentence summary of the meeting")

print("Signature fields:")
for name, field in MeetingSummary.model_fields.items():
    prefix = "INPUT" if field.json_schema_extra["__dspy_field_type"] == "input" else "OUTPUT"
    print(f"  [{prefix}] {name}: {field.annotation}")

## 6. First Prediction (Baseline)

Let's run our signature on a test example using `dspy.Predict` — the simplest DSPy module. This uses the signature's docstring and field descriptions as the prompt, with **no** few-shot examples and **no** optimization.

This is our baseline — the equivalent of a first-draft hand-written prompt.

In [ ]:
baseline = dspy.Predict(MeetingSummary)

# Run on one test example
test_example = testset[0]
result = baseline(transcript=test_example.transcript)

print("=== PREDICTED ===")
print(f"\nDecisions: {result.decisions}")
print(f"\nAction items: {result.action_items}")
print(f"\nSummary: {result.summary}")

print("\n=== EXPECTED ===")
print(f"\nDecisions: {test_example.decisions}")
print(f"\nAction items: {test_example.action_items}")
print(f"\nSummary: {test_example.summary}")

### Inspect the Auto-Generated Prompt

Let's see what DSPy actually sent to the model. Notice how it constructed a prompt from just the signature — no hand-writing needed.

In [ ]:
dspy.inspect_history(n=1)

## 7. Define a Quality Metric

To evaluate and optimize, we need a metric that scores how good a prediction is. Our composite metric checks two things:

1. **Keyword coverage (60%)** — do the predicted decisions and action items overlap with the expected ones? Uses token-level F1.
2. **Summary completeness (40%)** — does the summary mention key terms from the decisions?

This is lightweight and fast — no LLM-as-judge calls, so evaluation runs quickly.

In [ ]:
def tokenize(items):
    """Extract significant tokens from a list of strings."""
    tokens = set()
    for item in items:
        for word in item.lower().split():
            word = word.strip(".,;:()[]{}\"'")
            if len(word) > 2:
                tokens.add(word)
    return tokens


def keyword_f1(expected_items, predicted_items):
    """Token-level F1 between expected and predicted item lists."""
    expected_tokens = tokenize(expected_items)
    predicted_tokens = tokenize(predicted_items)
    if not expected_tokens or not predicted_tokens:
        return 0.0
    overlap = expected_tokens & predicted_tokens
    precision = len(overlap) / len(predicted_tokens)
    recall = len(overlap) / len(expected_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def summary_completeness(expected_decisions, summary):
    """Check if key decision terms appear in the summary."""
    if not summary or not expected_decisions:
        return 0.0
    summary_lower = summary.lower()
    hits = 0
    for decision in expected_decisions:
        words = [w.strip(".,;:()[]{}\"'").lower() for w in decision.split()]
        significant = [w for w in words if len(w) > 3]
        if any(w in summary_lower for w in significant):
            hits += 1
    return hits / len(expected_decisions)


def meeting_metric(example, prediction, trace=None):
    """Composite metric: keyword coverage (60%) + summary completeness (40%)."""
    # Combine decisions + action items for coverage scoring
    expected_all = example.decisions + example.action_items
    predicted_all = prediction.decisions + prediction.action_items

    coverage = keyword_f1(expected_all, predicted_all)
    completeness = summary_completeness(example.decisions, prediction.summary)

    score = 0.6 * coverage + 0.4 * completeness

    # For optimization: return bool for pass/fail
    if trace is not None:
        return score >= 0.4
    return score


# Test the metric on our earlier prediction
score = meeting_metric(test_example, result)
print(f"Score for our baseline prediction: {score:.3f}")

## 8. Baseline Evaluation

Now let's evaluate the baseline predictor across **all** test examples using `dspy.Evaluate`. This gives us a systematic "before" measurement — no more vibe-checking a few outputs by hand.

In [ ]:
evaluator = dspy.Evaluate(
    devset=testset,
    metric=meeting_metric,
    num_threads=2,
    display_progress=True,
    display_table=5,
)

baseline_score = evaluator(baseline)
print(f"\nBaseline average score: {baseline_score.score:.2f}")

## 9. Optimize with BootstrapFewShot

Here's where DSPy shines. Instead of manually writing few-shot examples or tweaking prompt wording, `BootstrapFewShot` does it automatically:

1. Runs the predictor on training examples
2. Scores each output with your metric
3. Selects the best input/output pairs as few-shot demonstrations
4. Compiles them into the prompt

The result: an optimized program with auto-selected demos that outperform the bare signature.

In [ ]:
optimizer = dspy.BootstrapFewShot(
    metric=meeting_metric,
    max_bootstrapped_demos=3,
    max_labeled_demos=3,
)

optimized = optimizer.compile(baseline, trainset=trainset)

# Evaluate the optimized version
optimized_score = evaluator(optimized)

print(f"\n{'='*40}")
print(f"Baseline score:  {baseline_score.score:.2f}")
print(f"Optimized score: {optimized_score.score:.2f}")
print(f"Improvement:     {optimized_score.score - baseline_score.score:+.2f}")
print(f"{'='*40}")

## 10. Inspect the Optimized Program

What did the optimizer actually produce? Use `dspy.inspect_history()` and look at the optimized program's demos to see the auto-generated few-shot examples.

In [ ]:
# Luke: ToDo — Inspect what the optimizer produced

## 11. Save and Load the Optimized Program

Optimized prompts are portable artifacts, save them to JSON and load them back.

In [ ]:
# Luke: ToDo — Save the optimized program and load it back


## 12. Try ChainOfThought

Swap `dspy.Predict` for `dspy.ChainOfThought` to add a reasoning step before the answer.

In [ ]:
# Luke: ToDo — Create a ChainOfThought predictor and evaluate it



## 13. Build a Custom Module

Wrap the signature in a `dspy.Module` subclass. This gives you a reusable, composable building block which serves as the foundation for multi-stage pipelines.

In [ ]:
# Luke: ToDo — Create a dspy.Module subclass



## 14. Improve the Metric with LLM-as-Judge

Our keyword-based metric is fast but shallow. Add an LLM-as-judge faithfulness check.


In [ ]:
# Luke: ToDo — Add a faithfulness judge to the metric



## 15. Compare All Approaches

Build a summary table comparing scores across all the approaches you've tried.


In [ ]:
# Luke: ToDo — Build a comparison table



## Conclusion

In this module you:

1. **Declared** a meeting summarization task as a typed DSPy signature — no prompt writing
2. **Measured** quality with a composite metric (keyword coverage + summary completeness)
3. **Optimized** prompts automatically with `BootstrapFewShot` — the optimizer found few-shot examples that improve quality with zero manual effort

### Key Takeaway

Prompts are **compiled artifacts**, not hand-written strings. Define what you want, measure quality, and let the optimizer do the work. This is how you scale prompt engineering from one-off experiments to production systems.

### Next Steps

- Try `MIPROv2` for joint instruction + demo optimization
- Build multi-stage pipelines by composing modules
- Deploy optimized programs as versioned artifacts in S3
- Integrate with CI/CD to gate deployments on quality thresholds

### Resources

- [DSPy Documentation](https://dspy.ai/)
- [DSPy Optimizers Guide](https://dspy.ai/learn/optimization/optimizers/)
- [Amazon Bedrock Documentation](https://docs.aws.amazon.com/bedrock/)